# RANST News Detection - Streaming Pipeline (Colab GPU)

Transcribeert gemeenteraad video, detecteert nieuws, koppelt echte sprekernamen.

**Drie lagen sprekernaamdetectie:**
1. **NotUBiz API** — Echte namen + partijen + timestamps (primair, 0 extra tijd)
2. **Transcript + OCR** — Regex op aankondigingen + naamkaart-OCR op videoframes (fallback, minimale extra tijd)
3. **Pyannote diarization** — AI spreker-clustering als basis (SPEAKER_XX labels)

## 1. SETUP

In [ ]:
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Geen GPU! Ga naar Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install -q "numpy<2.0" "scipy<1.14" openai-whisper pyannote.audio yt-dlp tqdm easyocr
!apt-get update -qq && apt-get install -y -qq ffmpeg 2>&1 | tail -1

## 2. CONFIG & IMPORTS

In [ ]:
import os
import json
import subprocess
import time as _time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Force install correct versions to solve both ModuleNotFoundError and Numpy compatibility
print("Installing dependencies and fixing version conflicts...")
!pip uninstall -y numpy numba
!pip install -q "numpy<2.0" "numba==0.60.0" "scipy<1.13.1" openai-whisper pyannote.audio

# Probeer de imports na installatie
import whisper
import torch
from tqdm import tqdm

try:
    from pyannote.audio import Pipeline
    DIARIZATION_AVAILABLE = True
except Exception as e:
    print(f'pyannote niet beschikbaar ({e}), diarization wordt overgeslagen')
    DIARIZATION_AVAILABLE = False

TEMP_DIR = '/content/temp'
OUTPUT_DIR = '/content/output'
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Global config variables
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FP16 = DEVICE == 'cuda'
WHISPER_MODEL = 'base'
NEWS_THRESHOLD = 0.65

# HuggingFace token (nodig voor pyannote speaker diarization)
from getpass import getpass
HF_TOKEN = getpass("Voer je HuggingFace token in (hf_...): ")
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN ingesteld')
else:
    print('Geen HF_TOKEN — pyannote diarization niet beschikbaar (NotUBiz+OCR werkt nog wel)')

print(f'Device: {DEVICE} | FP16: {FP16}')

## 3. CORE FUNCTIONS

In [ ]:
import whisper
import torch
import re
import cv2

_whisper_model = None
_diarizer_pipeline = None
_ocr_reader = None

def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
    return _whisper_model

def get_diarizer():
    global _diarizer_pipeline
    if not DIARIZATION_AVAILABLE:
        return None
    if _diarizer_pipeline is None:
        hf_token = os.getenv('HF_TOKEN', '')
        if not hf_token:
            return None
        from pyannote.audio import Pipeline
        _diarizer_pipeline = Pipeline.from_pretrained(
            'pyannote/speaker-diarization-3.1',
            use_auth_token=hf_token
        ).to(DEVICE)
    return _diarizer_pipeline

def get_ocr_reader():
    """Lazy-load EasyOCR reader (alleen als nodig, GPU-accelerated)."""
    global _ocr_reader
    if _ocr_reader is None:
        try:
            import easyocr
            _ocr_reader = easyocr.Reader(['nl', 'en'], gpu=torch.cuda.is_available(), verbose=False)
            print('OCR: EasyOCR geladen (GPU)' if torch.cuda.is_available() else 'OCR: EasyOCR geladen (CPU)')
        except Exception as e:
            print(f'OCR: EasyOCR niet beschikbaar ({e})')
            return None
    return _ocr_reader

def transcribe_chunk(audio_path):
    model = get_whisper_model()
    result = model.transcribe(
        audio_path, language='nl', task='transcribe',
        verbose=False, word_timestamps=False,
        fp16=FP16, temperature=0.3, beam_size=5
    )
    return result['segments']

# ── NotUBiz Speaker Timeline ─────────────────────────────────────────────
import json as _json
from urllib.request import Request as _Request, urlopen as _urlopen

NOTUBIZ_API = 'https://api.notubiz.nl'
NOTUBIZ_HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}

def _notubiz_get(path, timeout=20):
    req = _Request(f'{NOTUBIZ_API}{path}', headers=NOTUBIZ_HEADERS)
    with _urlopen(req, timeout=timeout) as resp:
        return _json.loads(resp.read().decode())

def _safe_list(obj):
    if isinstance(obj, list): return obj
    if isinstance(obj, dict): return [obj]
    return []

def find_notubiz_event(video_filename):
    """Zoek een NotUBiz event op basis van video bestandsnaam."""
    basename = os.path.splitext(os.path.basename(video_filename))[0]
    parts = basename.split()
    if len(parts) < 2:
        return None, None
    gemeente = parts[1] if len(parts) > 1 else None
    if not gemeente:
        return None, None
    print(f'NotUBiz: Zoeken naar gemeente "{gemeente}"...')
    try:
        data = _notubiz_get('/organisations')
        orgs = _safe_list(data.get('organisations', {}).get('organisation', []))
        org_id = None
        for org in orgs:
            name = org.get('name', '')
            if gemeente.lower() in name.lower():
                org_id = int(org.get('@attributes', {}).get('id', 0))
                print(f'NotUBiz: Gevonden org "{name}" (ID: {org_id})')
                break
        if not org_id:
            print(f'NotUBiz: Gemeente "{gemeente}" niet gevonden')
            return None, None
    except Exception as e:
        print(f'NotUBiz: Fout bij zoeken organisatie: {e}')
        return None, None
    target = os.path.basename(video_filename).lower().strip()
    try:
        data = _notubiz_get(f'/organisations/{org_id}/events')
        events = _safe_list(data.get('events', {}).get('event', []))
        for event in events:
            eid = int(event.get('@attributes', {}).get('id', 0))
            if not eid:
                continue
            try:
                detail = _notubiz_get(f'/events/{eid}')
                ev = _safe_list(detail.get('event', []))
                if not ev:
                    continue
                videos = _safe_list(ev[0].get('media', {}).get('video', []))
                for v in videos:
                    if v.get('filename', '').lower().strip() == target:
                        print(f'NotUBiz: Event gevonden! ID={eid} | {ev[0].get("title", "")}')
                        return eid, ev[0]
            except Exception:
                continue
    except Exception as e:
        print(f'NotUBiz: Fout bij zoeken events: {e}')
    print(f'NotUBiz: Geen event gevonden voor "{video_filename}"')
    return None, None

def fetch_speaker_timeline(event_id=None, event_data=None):
    if event_data is None:
        data = _notubiz_get(f'/events/{event_id}')
        event_data = _safe_list(data.get('event', []))[0]
    raw_speakers = _safe_list(event_data.get('speakers', {}).get('speaker', []))
    speakers = {}
    for s in raw_speakers:
        sid = s.get('@attributes', {}).get('id')
        if sid is None:
            continue
        sid = int(sid)
        name = s.get('name', '').strip().rstrip(',')
        party = s.get('party', {}).get('name', '')
        function = s.get('function', '')
        if party and party != 'Geen partij':
            label = f'{name} ({party})'
        elif function:
            label = f'{name} ({function})'
        else:
            label = name
        speakers[sid] = {'name': name, 'label': label, 'function': function, 'party': party}
    agenda_items = _safe_list(event_data.get('agenda', {}).get('agendaitem', []))
    raw_turns = []
    for item in agenda_items:
        indices = _safe_list(item.get('speaker_indexation', {}).get('speaker_index', []))
        for idx in indices:
            attrs = idx.get('@attributes', {})
            speaker_id = attrs.get('speaker_id')
            start_time = attrs.get('start_time')
            if speaker_id is not None and start_time is not None:
                raw_turns.append({'start_time': int(start_time), 'speaker_id': int(speaker_id)})
    raw_turns.sort(key=lambda x: x['start_time'])
    turns = []
    for i, turn in enumerate(raw_turns):
        sid = turn['speaker_id']
        info = speakers.get(sid, {'label': f'Spreker {sid}'})
        end_time = raw_turns[i + 1]['start_time'] if i + 1 < len(raw_turns) else None
        turns.append({
            'start_time': turn['start_time'],
            'end_time': end_time,
            'speaker_id': sid,
            'speaker_label': info['label'],
        })
    print(f'NotUBiz: {len(turns)} sprekerbeurten, {len(speakers)} sprekers')
    for sid, info in speakers.items():
        print(f'  • {info["label"]}')
    return {'speakers': speakers, 'turns': turns}

def match_speaker_from_timeline(segment_start_seconds, timeline):
    turns = timeline.get('turns', [])
    if not turns:
        return 'Spreker'
    for turn in turns:
        t_start = turn['start_time']
        t_end = turn['end_time']
        if t_end is None:
            if segment_start_seconds >= t_start:
                return turn['speaker_label']
        elif t_start <= segment_start_seconds < t_end:
            return turn['speaker_label']
    return 'Spreker'

# ── Transcript-gebaseerde naamherkenning ──────────────────────────────────
# Vangt aankondigingen op: "het woord is aan mevrouw Jansen",
# "dank u wel meneer De Vries", "voorzitter Bakker" etc.

_INTRO_PATTERNS = [
    # "het woord is aan (de heer/mevrouw) Voornaam Achternaam"
    r'(?:het woord is aan|ik geef het woord aan|het woord aan)\s+(?:de heer|mevrouw|mijnheer|dhr\.?|mevr\.?|mw\.?)?\s*([A-Z][a-zéèëïöüà]+(?:\s+(?:van|de|den|der|het|ter|ten)\s+)?[A-Z][a-zéèëïöüà]+)',
    # "dank u wel meneer/mevrouw Achternaam"
    r'(?:dank u wel|dankuwel|dank je wel|bedankt)\s+(?:de heer|mevrouw|meneer|mijnheer)?\s*([A-Z][a-zéèëïöüà]+(?:\s+(?:van|de|den|der|het|ter|ten)\s+)?[A-Z][a-zéèëïöüà]+)',
    # "voorzitter Naam" / "wethouder Naam" / "raadslid Naam"
    r'(?:voorzitter|wethouder|burgemeester|griffier|secretaris|raadslid|schepen)\s+([A-Z][a-zéèëïöüà]+(?:\s+(?:van|de|den|der|het|ter|ten)\s+)?[A-Z][a-zéèëïöüà]+)',
    # "de heer/mevrouw Achternaam" als los patroon
    r'(?:de heer|mevrouw|meneer|mijnheer)\s+([A-Z][a-zéèëïöüà]+(?:\s+(?:van|de|den|der|het|ter|ten)\s+)?[A-Z][a-zéèëïöüà]+)',
]

# Woorden die op namen lijken maar het niet zijn
_NAME_BLACKLIST = {
    'Europa', 'Nederland', 'Amsterdam', 'Rotterdam', 'Utrecht', 'Vlaanderen',
    'Belgie', 'Ranst', 'Antwerpen', 'Brussel', 'Gemeente', 'Commissie',
    'Raad', 'College', 'Kamer', 'Senaat', 'Groen Links', 'Partij',
}

def extract_names_from_transcript(text):
    """Haal sprekernamen uit getranscribeerde tekst via aankondigingspatronen.
    Returns: list van gevonden namen (strings).
    Kost 0 extra tijd — puur regex op bestaande transcriptie.
    """
    found = []
    for pattern in _INTRO_PATTERNS:
        for match in re.finditer(pattern, text):
            name = match.group(1).strip()
            if name and name not in _NAME_BLACKLIST and len(name) > 3:
                found.append(name)
    return found

# ── OCR naamdetectie uit videoframes ─────────────────────────────────────
# Bij een sprekerwisseling: pak 1 frame, scan de onderste 25% voor
# de naamkaart die vaak in beeld staat. Duurt ~0.3s per frame op GPU.

def ocr_name_from_frame(video_path, timestamp_seconds):
    """Pak 1 frame op timestamp, OCR de onderste 25% voor naamkaart.
    Returns: gevonden naam (str) of None.
    """
    reader = get_ocr_reader()
    if reader is None:
        return None
    try:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            cap.release()
            return None
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(timestamp_seconds * fps))
        ret, frame = cap.read()
        cap.release()
        if not ret or frame is None:
            return None

        h = frame.shape[0]
        # Alleen onderste 25% — daar staat de naamkaart
        bottom = frame[int(h * 0.75):, :]
        results = reader.readtext(bottom, detail=1, paragraph=False)

        # Filter op tekst die eruitziet als een naam (2+ woorden, eerste letter hoofdletter)
        for (bbox, text, conf) in results:
            text = text.strip()
            if conf < 0.4 or len(text) < 4:
                continue
            # Naamkaart check: bevat minstens 2 woorden met hoofdletters, of rol + naam
            words = text.split()
            if len(words) >= 2:
                caps = sum(1 for w in words if w[0].isupper())
                if caps >= 2 and text not in _NAME_BLACKLIST:
                    return text
        return None
    except Exception:
        return None

# ── Speaker Registry ─────────────────────────────────────────────────────
# Koppelt pyannote SPEAKER_XX labels aan echte namen via transcript + OCR.
# Bouwt zich op per chunk — geen extra processing pass nodig.

class SpeakerRegistry:
    """Mapt pyannote speaker labels (SPEAKER_00 etc.) naar echte namen.
    Leert namen via: transcript-aankondigingen, OCR-naamkaarten.
    """
    def __init__(self):
        self.label_to_name = {}      # SPEAKER_00 → "Jan de Vries"
        self.known_names = set()     # Alle gevonden namen
        self.pending_name = None     # Naam uit aankondiging, wacht op toewijzing

    def learn_from_transcript(self, text):
        """Zoek aangekondigde namen in transcript-tekst."""
        names = extract_names_from_transcript(text)
        for name in names:
            self.known_names.add(name)
            # Laatste aangekondigde naam → volgende spreker
            self.pending_name = name

    def learn_from_ocr(self, name):
        """Registreer naam gevonden via OCR."""
        if name:
            self.known_names.add(name)
            self.pending_name = name

    def resolve(self, pyannote_label):
        """Geef echte naam voor pyannote label, of het label zelf."""
        # Al eerder gekoppeld?
        if pyannote_label in self.label_to_name:
            return self.label_to_name[pyannote_label]

        # Pending naam toewijzen aan deze speaker
        if self.pending_name:
            self.label_to_name[pyannote_label] = self.pending_name
            name = self.pending_name
            self.pending_name = None
            return name

        return pyannote_label

    def summary(self):
        """Return dict van alle gevonden koppelingen."""
        return dict(self.label_to_name)

# ── Pyannote diarization fallback ────────────────────────────────────────

def diarize_chunk(audio_path):
    pipeline = get_diarizer()
    if pipeline is None:
        return []
    try:
        diarization = pipeline(audio_path)
        turns = []
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            turns.append({'speaker': speaker, 'start': round(turn.start, 2), 'end': round(turn.end, 2)})
        return turns
    except:
        return []

def match_speaker_to_segment(segment, turns):
    if not turns:
        return 'Spreker'
    seg_start = segment['start']
    for turn in turns:
        if turn['start'] <= seg_start < turn['end']:
            return turn['speaker']
    return 'Spreker'

# ── Nieuwsdetectie ───────────────────────────────────────────────────────

def detect_news(text):
    news_keywords = [
        'budget', 'besluit', 'wet', 'regel', 'voorstel', 'motie',
        'financieel', 'miljoen', 'bouw', 'project', 'belangrijk',
        'subsidie', 'belasting', 'stemming', 'amendement', 'bezwaar',
        'commissie', 'vergunning', 'bestemmingsplan', 'meerderheid'
    ]
    text_lower = text.lower()
    matches = [kw for kw in news_keywords if kw in text_lower]
    score = min(len(matches) * 0.15, 1.0)
    reason = f'Keywords: {chr(44).join(matches[:5])}' if matches else 'Geen keywords'
    return {
        'is_newsworthy': score > NEWS_THRESHOLD,
        'score': round(score, 2),
        'category': 'politiek',
        'reason': reason
    }

print('Functies geladen: Whisper, NotUBiz, OCR, Transcript-herkenning, SpeakerRegistry, Nieuws')

## 4. VIDEO SELECTEREN

In [ ]:
import os
from datetime import datetime

# --- Voer hier het pad naar je videobestand in ---
# Upload je MP4 naar /content/ via het Files-paneel (links)
# Of gebruik Google Drive: drive.mount('/content/drive')
video_path = '/content/04.03.26 Amsterdam WV HZ.mp4' # @param {type:"string"}

if not os.path.exists(video_path):
    mp4_files = [f for f in os.listdir('/content/') if f.endswith('.mp4')]
    if len(mp4_files) == 1:
        video_path = f'/content/{mp4_files[0]}'
        print(f'MP4-bestand gevonden: {os.path.basename(video_path)}')
    else:
        raise FileNotFoundError(
            f'Video niet gevonden: {video_path}\n'
            f'Upload een MP4 naar /content/ of pas video_path aan.'
        )

print(f'Te verwerken video: {os.path.basename(video_path)}')
print(f'Size: {os.path.getsize(video_path) / (1024*1024):.1f} MB')

video_entry = {'title': os.path.basename(video_path), 'duration': 0}
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

## 5. STREAMING PROCESSOR

In [ ]:
class StreamingProcessor:
    """Verwerkt video in chunks: transcriptie → sprekerkoppeling → nieuwsdetectie.

    Sprekernamen worden bepaald via drie lagen (van best naar fallback):
      1. NotUBiz API — exacte namen + partijen + timestamps (gratis, 0 extra tijd)
      2. Transcript-herkenning — regex op aankondigingen ("het woord is aan...") (0 extra tijd)
         + OCR op videoframes — naamkaart onderaan beeld (~0.3s per wisseling)
      3. Pyannote diarization — AI spreker-clustering (generieke labels SPEAKER_XX)
    """

    def __init__(self, video_path, notubiz_event_id=None):
        self.video_path = video_path
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.session_id = f'ranst_news_{self.timestamp}'
        self.all_segments = []
        self.news_alerts = []
        self.output_lock = threading.Lock()
        self._start_time = None
        self._last_update_time = None
        self.speaker_source = 'pyannote'

        # Speaker Registry voor transcript+OCR naamkoppeling
        self.speaker_registry = SpeakerRegistry()

        # ── Laag 1: NotUBiz sprekerstijdlijn ophalen ──
        self.speaker_timeline = None
        self.use_notubiz = False
        try:
            if notubiz_event_id:
                print(f'NotUBiz: Sprekersdata ophalen voor event {notubiz_event_id}...')
                self.speaker_timeline = fetch_speaker_timeline(event_id=notubiz_event_id)
                self.use_notubiz = True
                self.speaker_source = 'notubiz'
            else:
                print(f'NotUBiz: Automatisch zoeken naar event voor "{os.path.basename(video_path)}"...')
                eid, event_data = find_notubiz_event(video_path)
                if eid and event_data:
                    self.speaker_timeline = fetch_speaker_timeline(event_data=event_data)
                    self.use_notubiz = True
                    self.speaker_source = 'notubiz'
                else:
                    print('NotUBiz: Geen data — Laag 2 (transcript+OCR) + Laag 3 (pyannote) actief')
        except Exception as e:
            print(f'NotUBiz: Fout ({e}) — Laag 2 (transcript+OCR) + Laag 3 (pyannote) actief')

    def split_into_chunks(self, chunk_minutes=5):
        ffprobe_command = [
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            self.video_path
        ]
        result = subprocess.run(ffprobe_command, capture_output=True, text=True, check=False)
        if result.returncode != 0:
            raise RuntimeError(f"ffprobe failed: {result.stderr.strip()}")
        duration_str = result.stdout.strip()
        if not duration_str:
            raise ValueError("ffprobe returned empty duration.")
        duration = float(duration_str)
        chunk_size = chunk_minutes * 60
        chunks = []
        for i in range(0, int(duration), int(chunk_size)):
            chunks.append({
                'num': len(chunks) + 1,
                'start': i,
                'end': min(i + chunk_size, duration),
                'duration': min(chunk_size, duration - i)
            })
        return chunks

    def extract_chunk(self, chunk_info):
        num = chunk_info['num']
        out_file = f'{TEMP_DIR}/chunk_{num:03d}.wav'
        subprocess.run([
            'ffmpeg', '-y', '-i', self.video_path,
            '-ss', str(chunk_info['start']),
            '-t', str(chunk_info['duration']),
            '-ac', '1', '-ar', '16000', '-q:a', '9', out_file
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return out_file if os.path.exists(out_file) else None

    def process_chunk(self, chunk_info, audio_file):
        try:
            segments = transcribe_chunk(audio_file)
            if not segments:
                return {'alert': None, 'segments': []}

            # ── Laag 1: NotUBiz (exacte namen + timestamps) ──
            if self.use_notubiz and self.speaker_timeline:
                enriched = []
                for seg in segments:
                    abs_start = chunk_info['start'] + seg['start']
                    speaker = match_speaker_from_timeline(abs_start, self.speaker_timeline)
                    enriched.append({
                        'chunk': chunk_info['num'],
                        'speaker': speaker,
                        'text': seg['text'].strip(),
                        'start': abs_start,
                        'end': chunk_info['start'] + seg['end']
                    })

            # ── Laag 2+3: Transcript+OCR naamherkenning + pyannote ──
            else:
                turns = diarize_chunk(audio_file)
                enriched = []
                prev_pyannote_label = None

                for seg in segments:
                    abs_start = chunk_info['start'] + seg['start']
                    text = seg['text'].strip()
                    pyannote_label = match_speaker_to_segment(seg, turns)

                    # Laag 2a: Zoek namen in transcript-tekst (0 extra tijd)
                    self.speaker_registry.learn_from_transcript(text)

                    # Laag 2b: OCR bij sprekerwisseling (~0.3s per keer, alleen bij wisseling)
                    if pyannote_label != prev_pyannote_label:
                        ocr_name = ocr_name_from_frame(self.video_path, abs_start)
                        if ocr_name:
                            self.speaker_registry.learn_from_ocr(ocr_name)
                    prev_pyannote_label = pyannote_label

                    # Resolve: geef echte naam als die bekend is
                    speaker = self.speaker_registry.resolve(pyannote_label)

                    enriched.append({
                        'chunk': chunk_info['num'],
                        'speaker': speaker,
                        'text': text,
                        'start': abs_start,
                        'end': chunk_info['start'] + seg['end']
                    })

                # Update speaker_source
                if self.speaker_registry.known_names:
                    self.speaker_source = 'transcript+ocr'

            chunk_text = ' '.join([s['text'] for s in enriched])
            news = detect_news(chunk_text)
            alert = None
            if news['is_newsworthy']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'chunk': chunk_info['num'],
                    'time': f"{chunk_info['start']/60:.0f}m-{chunk_info['end']/60:.0f}m",
                    'score': news['score'],
                    'category': news['category'],
                    'reason': news['reason'],
                    'speakers': list(set([s['speaker'] for s in enriched])),
                    'text_preview': chunk_text[:300]
                }
            return {'alert': alert, 'segments': enriched}
        except Exception as e:
            print(f'  Chunk {chunk_info["num"]} error: {e}')
            return {'alert': None, 'segments': []}

    def _print_progress(self, chunks_done, total_chunks):
        now = _time.time()
        elapsed = now - self._start_time
        pct = (chunks_done / total_chunks) * 100
        if chunks_done > 0:
            per_chunk = elapsed / chunks_done
            remaining = per_chunk * (total_chunks - chunks_done)
            print(f'VOORTGANG: {pct:.0f}% ({chunks_done}/{total_chunks}) - Resterend: {remaining/60:.1f} min')
        self._last_update_time = now

    def process_streaming(self):
        print('=' * 60)
        print('STREAMING PIPELINE START')
        print(f'Speaker detectie: {"NotUBiz API" if self.use_notubiz else "Transcript+OCR+Pyannote"}')
        print('=' * 60)
        self._start_time = _time.time()
        self._last_update_time = self._start_time
        chunks = self.split_into_chunks()
        total_chunks = len(chunks)
        print(f'{total_chunks} chunks van 5 minuten te verwerken')

        # Extractie parallel (I/O-bound)
        chunk_files = {}
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(self.extract_chunk, c): c for c in chunks}
            for future in tqdm(as_completed(futures), total=total_chunks, desc='Audio extractie'):
                chunk = futures[future]
                chunk_files[chunk['num']] = future.result()

        # Verwerking sequentieel (GPU-bound)
        chunks_done = 0
        for chunk in chunks:
            if chunk['num'] in chunk_files and chunk_files[chunk['num']]:
                result = self.process_chunk(chunk, chunk_files[chunk['num']])
                with self.output_lock:
                    self.all_segments.extend(result['segments'])
                    if result['alert']:
                        self.news_alerts.append(result['alert'])
                try:
                    os.remove(chunk_files[chunk['num']])
                except:
                    pass
            chunks_done += 1
            self._print_progress(chunks_done, total_chunks)

        # Post-processing: hernoem eerder verwerkte segmenten met later geleerde namen
        if not self.use_notubiz and self.speaker_registry.label_to_name:
            mapping = self.speaker_registry.label_to_name
            renamed = 0
            for seg in self.all_segments:
                old = seg['speaker']
                if old in mapping and seg['speaker'] != mapping[old]:
                    seg['speaker'] = mapping[old]
                    renamed += 1
            if renamed:
                print(f'Post-processing: {renamed} segmenten herbenoemd met geleerde namen')

        elapsed = _time.time() - self._start_time
        print(f'\nKlaar in {elapsed/60:.1f} minuten')
        self.save_results()

    def save_results(self):
        elapsed = _time.time() - self._start_time if self._start_time else 0

        # JSON output
        output_file_json = f'{OUTPUT_DIR}/{self.session_id}.json'
        output_data = {
            'metadata': {
                'video': os.path.basename(self.video_path),
                'total_segments': len(self.all_segments),
                'total_alerts': len(self.news_alerts),
                'processing_time_minutes': round(elapsed / 60, 1),
                'speaker_source': self.speaker_source,
            },
            'segments': self.all_segments,
            'news_alerts': self.news_alerts,
        }
        if self.speaker_timeline:
            output_data['speakers'] = {
                str(k): v for k, v in self.speaker_timeline.get('speakers', {}).items()
            }
        if self.speaker_registry.known_names:
            output_data['learned_names'] = list(self.speaker_registry.known_names)
            output_data['speaker_mapping'] = self.speaker_registry.summary()

        with open(output_file_json, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        # Leesbaar transcript
        transcript_file = f'{OUTPUT_DIR}/{self.session_id}_transcript.txt'
        with open(transcript_file, 'w', encoding='utf-8') as f:
            f.write(f"TRANSCRIPT: {os.path.basename(self.video_path)}\n")
            f.write(f"GEGENEREERD OP: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"SPREKERSBRON: {self.speaker_source}\n")
            f.write(f"VERWERKING: {elapsed/60:.1f} minuten\n")
            f.write("=" * 60 + "\n")

            # Sprekerslijst
            all_speakers = set()
            if self.speaker_timeline:
                for s in self.speaker_timeline.get('speakers', {}).values():
                    all_speakers.add(s.get('label', s.get('name', '')))
            if self.speaker_registry.known_names:
                all_speakers.update(self.speaker_registry.known_names)
            if not all_speakers:
                all_speakers = set(s.get('speaker', '') for s in self.all_segments)

            f.write(f"\nSPREKERS ({len(all_speakers)}):\n")
            for name in sorted(all_speakers):
                if name and name != 'Spreker':
                    f.write(f"  - {name}\n")
            f.write("\n" + "-" * 60 + "\n\n")

            # Transcript
            prev_speaker = None
            for seg in self.all_segments:
                ts = f"[{int(seg['start']//60):02d}:{int(seg['start']%60):02d}]"
                speaker = seg.get('speaker', 'Spreker')
                if speaker != prev_speaker:
                    f.write(f"\n{ts} {speaker}:\n")
                    prev_speaker = speaker
                f.write(f"  {seg['text']}\n")

        print(f'Opgeslagen: {transcript_file}')
        print(f'Opgeslagen: {output_file_json}')

## 6. RUN

In [ ]:
# Optioneel: stel het NotUBiz event ID in als je die al weet.
# Laat op None voor automatische detectie via bestandsnaam.
notubiz_event_id = None  # @param {type:"raw"}
# Voorbeeld: notubiz_event_id = 1462423  # WV Hoorzitting liftstoringen Amsterdam

print(f'Video: {video_path}')
print(f'Output: {OUTPUT_DIR}\\n')

processor = StreamingProcessor(video_path, notubiz_event_id=notubiz_event_id)
processor.process_streaming()

## 7. DOWNLOAD RESULTATEN

In [ ]:
from google.colab import files
import os
import time

OUTPUT_DIR = '/content/output'

if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    # Haal alle bestanden op (zowel .json als .txt)
    output_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('ranst_news_')]

    if not output_files:
        print('Geen tekst- of JSON-resultaten gevonden.')
    else:
        print(f'{len(output_files)} bestanden gevonden. Downloaden start nu...')
        for filename in sorted(output_files):
            path = os.path.join(OUTPUT_DIR, filename)
            print(f'Downloaden: {filename}')
            files.download(path)
            # Kleine pauze om te voorkomen dat de browser meerdere downloads blokkeert
            time.sleep(1)
        print('\nDownload-acties uitgevoerd. Controleer je downloadmap op de .txt bestanden!')
else:
    print('De output map is leeg.')

## 8. PREVIEW RESULTATEN

In [ ]:
latest_json = f'{OUTPUT_DIR}/{processor.session_id}.json'
with open(latest_json) as f:
    results = json.load(f)

meta = results.get('metadata', {})
print(f'\n{"="*60}')
print('RESULTATEN SAMENVATTING')
print(f'{"="*60}')
print(f'Video: {meta.get("video", "?")}')
print(f'Segmenten: {meta.get("total_segments", len(results.get("segments", [])))}')
print(f'Nieuws alerts: {meta.get("total_alerts", len(results.get("news_alerts", [])))}')
print(f'Verwerkingstijd: {meta.get("processing_time_minutes", "?")} min')
print(f'Sprekersbron: {meta.get("speaker_source", "?")}')

if results.get('learned_names'):
    print(f'\nGeleerde namen ({len(results["learned_names"])}): {", ".join(results["learned_names"])}')
if results.get('speaker_mapping'):
    print('\nSpreker-mapping:')
    for label, name in results['speaker_mapping'].items():
        print(f'  {label} → {name}')

if results.get('news_alerts'):
    print(f'\nTOP ALERTS:')
    for i, alert in enumerate(results['news_alerts'][:5], 1):
        print(f'\n  {i}. [{alert["time"]}] Score: {alert["score"]:.2f}')
        print(f'     {alert["reason"]}')
        speakers_str = ', '.join(alert['speakers'])
        print(f'     Sprekers: {speakers_str}')
        print(f'     Preview: {alert["text_preview"][:100]}...')

## 9. SHUTDOWN (GPU vrijgeven)

In [ ]:
# Voer ALLEEN uit als je klaar bent met downloaden!
import time
print('Runtime wordt over 10 seconden afgesloten...')
time.sleep(10)
os.system('kill -9 -1')